# Noisy grid: RandomDefect vs TitForTat

An 11 × 11 sweep over two axes:

- **Rows (`i`)**: number of `RandomDefect` players, `count = 10 * i` for `i ∈ {0..10}`.
  The remaining `100 - 10*i` players are `TitForTat`. So `i=0` is a pure
  TFT population and `i=10` is a pure RandomDefect population.
- **Columns (`j`)**: defection probability of every `RandomDefect` player
  in that game, `p = 0.1 * j` for `j ∈ {0..10}`. `j=0` makes them
  effectively AlwaysCooperate, `j=10` makes them AlwaysDefect.

Each of the 121 games runs for 100 rounds. Every game holds 100 players,
so a single game plays `C(100,2) * 100 = 495 000` deals; the full grid
is about 60 million deals.

**Reproducibility:** a single `master_seed` seeds a `random.Random`
that produces per-cell seeds and per-player seeds. Same `master_seed`
→ identical grid.

**Collector:** for each finished game we record aggregates over the
per-player total scores: `sum`, `avg`, `max`, and the p95 / p90 / p80 /
p60 / p20 / p5 percentiles. At the end we draw one heatmap per metric
on the (i, j) grid.

## CONFIG

Change `master_seed`, `rounds`, or the axis lengths here. Everything
below re-derives from these three numbers.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
from pd import (
    DealPayoff,
    Game,
    Multigame,
    RandomDefect,
    ScoreStatsCollector,
    StaticGenerator,
    TitForTat,
)
from pd.collectors.score_stats import METRICS, PERCENTILES

In [ ]:
MASTER_SEED = 20260703
ROUNDS = 100
GRID_SIZE = 11  # i, j in range(11)
PLAYERS_PER_GAME = 100
STEP = 10  # 10 * i RandomDefect players
P_STEP = 0.1  # 0.1 * j defection probability

master_rng = random.Random(MASTER_SEED)


def _seed() -> int:
    # Draw a full 63-bit seed from the master RNG so per-object streams
    # are well-separated and reproducible.
    return master_rng.getrandbits(63)

# Classic Axelrod payoff (T=5, R=3, P=1, S=0). Change this to sweep
# over other payoff structures.
PAYOFF = DealPayoff(
    payoff_cc_1=3.0, payoff_cc_2=3.0,
    payoff_cd_1=0.0, payoff_cd_2=5.0,
    payoff_dc_1=5.0, payoff_dc_2=0.0,
    payoff_dd_1=1.0, payoff_dd_2=1.0,
)


## Build the 11 x 11 grid of games

In [ ]:
def build_game(i: int, j: int) -> Game:
    n_defect = STEP * i               # 0, 10, ..., 100
    n_tft = PLAYERS_PER_GAME - n_defect
    p = P_STEP * j                    # 0.0, 0.1, ..., 1.0

    players = []
    # Each RandomDefect gets its own independent stream so their noise
    # is not synchronized within a game.
    for _ in range(n_defect):
        players.append(RandomDefect(p=p, rng=random.Random(_seed())))
    for _ in range(n_tft):
        players.append(TitForTat())

    return Game(
        deal_generator=StaticGenerator(PAYOFF),
        players=players,
        total_rounds=ROUNDS,
        rng=random.Random(_seed()),
    )


grid = [
    [
        (
            build_game(i, j),
            {"i": i, "j": j, "n_defect": STEP * i, "p": P_STEP * j},
        )
        for j in range(GRID_SIZE)
    ]
    for i in range(GRID_SIZE)
]

print(f"built grid: {GRID_SIZE}x{GRID_SIZE} = {GRID_SIZE * GRID_SIZE} games, {ROUNDS} rounds each")

## Collector: score aggregates per game

In [ ]:
# ScoreStatsCollector is defined in pd.collectors.score_stats so
# ProcessPoolExecutor workers can pickle-import it. Same class,
# same metrics -- METRICS and PERCENTILES imported above.
collector = ScoreStatsCollector()

## Run the tournament grid

In [ ]:
# run_parallel uses ProcessPoolExecutor -- one game per worker.
# max_workers=None lets it default to os.cpu_count(). Rows arrive
# in completion order, not row-major; each GameStats carries its
# own (i, j) coordinates.
#
# If you hit BrokenProcessPool on Windows/Jupyter, try:
#   1. Restart the kernel before running this cell.
#   2. Multigame(grid, collector).run_parallel(max_workers=1)
#      -- forces a single worker so real tracebacks surface.
#   3. Multigame(grid, collector).run_parallel(mp_context="spawn")
Multigame(grid, collector).run_parallel(max_workers=None)
print(f"collected {len(collector.rows)} rows (expected {GRID_SIZE * GRID_SIZE})")


## Reshape results into (i, j) matrices

In [ ]:
def metric_matrix(name: str) -> np.ndarray:
    m = np.full((GRID_SIZE, GRID_SIZE), np.nan)
    for row in collector.rows:
        m[row.i, row.j] = row.metrics[name]
    return m


matrices = {name: metric_matrix(name) for name in METRICS}
for name, m in matrices.items():
    print(f"{name:>4}: min={m.min():10.2f}  max={m.max():10.2f}  mean={m.mean():10.2f}")

## Heatmaps for every metric

In [ ]:
def plot_heatmap(ax, matrix: np.ndarray, title: str) -> None:
    im = ax.imshow(matrix, origin="lower", aspect="auto", cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("j  (p = 0.1 * j)")
    ax.set_ylabel("i  (n_defect = 10 * i)")
    ax.set_xticks(range(GRID_SIZE))
    ax.set_yticks(range(GRID_SIZE))
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)


n = len(METRICS)                       # 9 metrics
cols = 3
rows = (n + cols - 1) // cols          # 3 rows
fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 3.8 * rows))
axes = axes.ravel()

for ax, name in zip(axes, METRICS):
    plot_heatmap(ax, matrices[name], name)
for ax in axes[n:]:
    ax.axis("off")

fig.suptitle(
    f"Score aggregates over {GRID_SIZE}x{GRID_SIZE} grid "
    f"(seed={MASTER_SEED}, rounds={ROUNDS}, {PLAYERS_PER_GAME} players/game)",
    y=1.02,
)
fig.tight_layout()
plt.show()